## Demonstração do Controller de Meta Financeira

Este notebook demonstra o funcionamento do controller de metas financeiras da API, cobrindo as seguintes operações:

- Criar meta financeira (`POST /users/create`)
- Buscar todos os metas financeiras ativos (`GET /users/all`)
- Buscar meta financeira por ID (`GET /users/{id}`)
- Atualizar meta financeira (`PATCH /users/{id}/update`)
- Desativar meta financeira (`DELETE /users/{id}/delete`)
- Restaurar meta financeira desativado (`POST /users/{id}/restore`)
- Deletar permanentemente do banco (`DELETE /users/{id}/force-delete`)


## Setup do Teste com FastAPI e TestClient

Import e configuração do FastAPI com o router `users`.

In [2]:
from fastapi.testclient import TestClient
from fastapi import FastAPI

from user.controller import users_router
from financial_goals.controller import financial_goals_router

app = FastAPI()
app.include_router(users_router)
app.include_router(financial_goals_router)

client = TestClient(app)

## Criar Usuário de Teste

**Endpoint:** 
`POST /users/create`  

**Descrição:** Cria um novo Usuário com os dados fornecidos no payload.

In [5]:
user_payload = {
    "first_name": "Leslie",
    "last_name": "Chow",
    "cpf": "462.216.300-45",
    "email": "lchow@yahoo.com",
    "password": "Strong@Password1976",
    "manual_balance": 12500.0
}

response = client.post("/users/create", json=user_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_user = response.json()
user_id = created_user["id"]

2025-06-15 14:28:00,289 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:28:00,299 INFO sqlalchemy.engine.Engine INSERT INTO users (id, first_name, last_name, email, cpf, password, manual_balance, created_at, updated_at, deleted_at) VALUES (%(id)s, %(first_name)s, %(last_name)s, %(email)s, %(cpf)s, %(password)s, %(manual_balance)s, %(created_at)s, %(updated_at)s, %(deleted_at)s)
2025-06-15 14:28:00,301 INFO sqlalchemy.engine.Engine [cached since 506.2s ago] {'id': '0684f02a04cb718780007743ae36150d', 'first_name': 'Leslie', 'last_name': 'Chow', 'email': 'lchow@yahoo.com', 'cpf': '46221630045', 'password': '$2b$12$8hoWXdP83eDopXV3IEmGPuTV1E8NG0kqwtbfa5EGZjWltfcya5UfC', 'manual_balance': 12500.0, 'created_at': datetime.datetime(2025, 6, 15, 14, 19, 8, 278163), 'updated_at': datetime.datetime(2025, 6, 15, 14, 19, 8, 278163), 'deleted_at': None}
2025-06-15 14:28:00,302 INFO sqlalchemy.engine.Engine ROLLBACK


IntegrityError: (pymysql.err.IntegrityError) (1062, "Duplicate entry 'lchow@yahoo.com' for key 'users.email'")
[SQL: INSERT INTO users (id, first_name, last_name, email, cpf, password, manual_balance, created_at, updated_at, deleted_at) VALUES (%(id)s, %(first_name)s, %(last_name)s, %(email)s, %(cpf)s, %(password)s, %(manual_balance)s, %(created_at)s, %(updated_at)s, %(deleted_at)s)]
[parameters: {'id': '0684f02a04cb718780007743ae36150d', 'first_name': 'Leslie', 'last_name': 'Chow', 'email': 'lchow@yahoo.com', 'cpf': '46221630045', 'password': '$2b$12$8hoWXdP83eDopXV3IEmGPuTV1E8NG0kqwtbfa5EGZjWltfcya5UfC', 'manual_balance': 12500.0, 'created_at': datetime.datetime(2025, 6, 15, 14, 19, 8, 278163), 'updated_at': datetime.datetime(2025, 6, 15, 14, 19, 8, 278163), 'deleted_at': None}]
(Background on this error at: https://sqlalche.me/e/20/gkpj)

## Criar Meta Financeira de Teste

**Endpoint:** 
`POST /financial_goals/create`  

**Descrição:** Cria uma nova Meta Financeira com os dados fornecidos no payload.

In [7]:
financial_goals_payload = {
    "user_id": user_id,
    "name": "Viagem a Las Vegas com o Alan",
    "description": "Comemorar a despedida de Solteiro dele!!!",
    "target_amount": 20000,
    "deadline": "2026-01-15"
}

response = client.post("/financial_goals/create", json=financial_goals_payload)
print("Status:", response.status_code)
print("Resposta:", response.json())
created_financial_goal = response.json()
financial_goal_id = created_financial_goal["id"]

2025-06-15 14:31:09,190 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:31:09,197 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.first_name AS users_first_name, users.last_name AS users_last_name, users.email AS users_email, users.cpf AS users_cpf, users.password AS users_password, users.manual_balance AS users_manual_balance, users.created_at AS users_created_at, users.updated_at AS users_updated_at, users.deleted_at AS users_deleted_at 
FROM users 
WHERE users.id = %(id_1)s AND users.deleted_at IS NULL 
 LIMIT %(param_1)s
2025-06-15 14:31:09,199 INFO sqlalchemy.engine.Engine [generated in 0.00177s] {'id_1': '0684f00a61a776ef8000c2a4a6427b9a', 'param_1': 1}
2025-06-15 14:31:09,203 INFO sqlalchemy.engine.Engine INSERT INTO financial_goals (id, user_id, name, description, target_amount, deadline, created_at, updated_at) VALUES (%(id)s, %(user_id)s, %(name)s, %(description)s, %(target_amount)s, %(deadline)s, %(created_at)s, %(updated_at)s)
2025-06-15 14:31:

## Buscar Todas as Metas Financeiras

**Endpoint:** `GET /financial_goals/all`  
**Descrição:** Retorna todos os metas financeiras no sistema.

In [8]:
response = client.get("/financial_goals/all")
print("Status:", response.status_code)
for user in response.json():
    print(user)

2025-06-15 14:31:13,684 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:31:13,687 INFO sqlalchemy.engine.Engine SELECT financial_goals.id AS financial_goals_id, financial_goals.user_id AS financial_goals_user_id, financial_goals.name AS financial_goals_name, financial_goals.description AS financial_goals_description, financial_goals.target_amount AS financial_goals_target_amount, financial_goals.deadline AS financial_goals_deadline, financial_goals.created_at AS financial_goals_created_at, financial_goals.updated_at AS financial_goals_updated_at 
FROM financial_goals
2025-06-15 14:31:13,689 INFO sqlalchemy.engine.Engine [cached since 674.1s ago] {}
2025-06-15 14:31:13,696 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 200
{'id': '06813c93-95b8-76e7-8000-990ef1776e03', 'user_id': '027339ad-5777-2ca2-8675-0a3a2ad47b31', 'name': 'Alcançar 100 mil reais até Dezembro-2025', 'description': None, 'target_amount': 100000.0, 'deadline': '2025-12-25', 'created_at': '2025-05-01T16:18

## Buscar meta financeira por ID

**Endpoint:** `GET /financial_goals/{id}`  
**Descrição:** Retorna os dados de uma Meta Financeira específica, identificada pelo ID.

In [9]:
response = client.get(f"/financial_goals/{financial_goal_id}")
print("Status:", response.status_code)
print("Meta Financeira:", response.json())

2025-06-15 14:31:18,125 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:31:18,130 INFO sqlalchemy.engine.Engine SELECT financial_goals.id AS financial_goals_id, financial_goals.user_id AS financial_goals_user_id, financial_goals.name AS financial_goals_name, financial_goals.description AS financial_goals_description, financial_goals.target_amount AS financial_goals_target_amount, financial_goals.deadline AS financial_goals_deadline, financial_goals.created_at AS financial_goals_created_at, financial_goals.updated_at AS financial_goals_updated_at 
FROM financial_goals 
WHERE financial_goals.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-15 14:31:18,130 INFO sqlalchemy.engine.Engine [generated in 0.00064s] {'id_1': '0684f035d341734e800083210cf41c11', 'param_1': 1}
2025-06-15 14:31:18,134 INFO sqlalchemy.engine.Engine ROLLBACK
Status: 200
Meta Financeira: {'id': '0684f035-d341-734e-8000-83210cf41c11', 'user_id': '0684f00a-61a7-76ef-8000-c2a4a6427b9a', 'name': 'Viagem a Las Vegas c

## Atualizar Dados do meta financeira

**Endpoint:** `PATCH /users/{id}/update`  
**Descrição:** Atualiza os campos fornecidos do meta financeira (ex: saldo manual).

In [10]:
update_payload = {
    "deadline": "2025-12-25"
}

response = client.patch(f"/financial_goals/{financial_goal_id}/update", json=update_payload)
print("Status:", response.status_code)
print("Atualizado:", response.json())

2025-06-15 14:31:24,191 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:31:24,196 INFO sqlalchemy.engine.Engine SELECT financial_goals.id AS financial_goals_id, financial_goals.user_id AS financial_goals_user_id, financial_goals.name AS financial_goals_name, financial_goals.description AS financial_goals_description, financial_goals.target_amount AS financial_goals_target_amount, financial_goals.deadline AS financial_goals_deadline, financial_goals.created_at AS financial_goals_created_at, financial_goals.updated_at AS financial_goals_updated_at 
FROM financial_goals 
WHERE financial_goals.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-15 14:31:24,197 INFO sqlalchemy.engine.Engine [cached since 6.067s ago] {'id_1': '0684f035d341734e800083210cf41c11', 'param_1': 1}
2025-06-15 14:31:24,200 INFO sqlalchemy.engine.Engine SELECT financial_goals.id AS financial_goals_id, financial_goals.user_id AS financial_goals_user_id, financial_goals.name AS financial_goals_name, financial_goals.

## Deletar Meta Financeira Permanentemente

**Endpoint:** `DELETE /financial_goals/{financial_goal_id}/delete`  
**Descrição:** Deleta permanentemente a Meta Financeira do banco de dados

In [11]:
response = client.delete(f"/financial_goals/{financial_goal_id}/delete")
print("Status:", response.status_code)
print("Forçando a Deleção no banco de dados:", response.json())

2025-06-15 14:31:29,800 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-06-15 14:31:29,806 INFO sqlalchemy.engine.Engine SELECT financial_goals.id AS financial_goals_id, financial_goals.user_id AS financial_goals_user_id, financial_goals.name AS financial_goals_name, financial_goals.description AS financial_goals_description, financial_goals.target_amount AS financial_goals_target_amount, financial_goals.deadline AS financial_goals_deadline, financial_goals.created_at AS financial_goals_created_at, financial_goals.updated_at AS financial_goals_updated_at 
FROM financial_goals 
WHERE financial_goals.id = %(id_1)s 
 LIMIT %(param_1)s
2025-06-15 14:31:29,807 INFO sqlalchemy.engine.Engine [cached since 11.68s ago] {'id_1': '0684f035d341734e800083210cf41c11', 'param_1': 1}
2025-06-15 14:31:29,809 INFO sqlalchemy.engine.Engine SELECT financial_goals.id AS financial_goals_id, financial_goals.user_id AS financial_goals_user_id, financial_goals.name AS financial_goals_name, financial_goals.

JSONDecodeError: Expecting value: line 1 column 1 (char 0)